---
title: "读取 LakeSoul 机器人轨迹与训练样本"
format: docusaurus-md
execute:
  enabled: false
---

本示例读取已有的 **Prepared 表**：展示一个时间步的状态、动作和四路相机图片，再调用 LakeSoul API 构造一个训练窗口。
Prepared 已完成质量筛选与数据集划分，包含 `split` 和 `valid_interval_id`；本示例不重新执行 QC、不写表、不训练模型。

使用逐帧 JPEG 图片布局（不是 GOP 表）。在配置好 LakeSoul PG 和 S3 环境变量的 Notebook 环境中依次运行下面四个单元格，然后保存输出。依赖：`lakesoul[daft]`、`pillow`、`numpy`、`ipykernel`。


## 1. 连接已有 Prepared 表

修改表名即可使用之前生成的结果。`EPISODE_ID` 指定要查看的轨迹；本示例选择四路相机。连接参数从环境读取，不在输出中展示。


In [ ]:
import os
from io import BytesIO

import numpy as np
import pyarrow.dataset as ds
from PIL import Image
from IPython.display import display
from lakesoul import LakeSoulCatalog
from lakesoul.embodied.daft import read_samples

TABLE = "aloha_velcro_prepared_4cam_ann"  # 改成你的 Prepared 表名
NAMESPACE = "default"
EPISODE_ID = "ep000000"
CAMERAS = ["cam_high", "cam_low", "cam_left_wrist", "cam_right_wrist"]

catalog = LakeSoulCatalog.from_env(
    namespace=NAMESPACE,
    object_store_options={
        "fs.s3a.endpoint": os.environ["AWS_ENDPOINT"],
        "fs.s3a.access.key": os.environ["AWS_ACCESS_KEY_ID"],
        "fs.s3a.secret.key": os.environ["AWS_SECRET_ACCESS_KEY"],
        "fs.s3a.path.style.access": "true",
    },
)
table = catalog.table(TABLE)
print(f"Prepared 表：{NAMESPACE}.{TABLE}")


## 2. 查询一条轨迹的第一条保留记录

按原始帧号排序，只取一行用于展示。被筛掉的帧不会出现在 Prepared 中，因此第一条记录不一定是第 0 帧。


In [ ]:
episode_scan = table.scan().filter(ds.field("episode_id") == EPISODE_ID)
rows = (
    episode_scan.to_daft()
    .sort("frame_index")
    .limit(1)
    .to_arrow()
    .to_pylist()
)
if not rows:
    raise ValueError(f"Prepared 中没有轨迹 {EPISODE_ID}")
row = rows[0]

print("episode:", row["episode_id"])
print("frame_index:", row["frame_index"])
print("timestamp:", round(row["timestamp"], 4), "s")
print("split:", row["split"])
print("valid_interval_id:", row["valid_interval_id"])
print("state:", np.round(row["observation_state"], 3))
print("action:", np.round(row["action"], 3))


## 3. 查看同一时刻的四路相机图片

将 JPEG 字节解码为静态图片。四张图片来自同一条记录；仅缩小显示尺寸，不修改表内图像。静态图片可以随 Notebook 输出导出到文档。


In [ ]:
for camera in CAMERAS:
    print(f"{camera} | frame {row['frame_index']}")
    with Image.open(BytesIO(row[camera])) as image:
        preview = image.convert("RGB")
        preview.thumbnail((480, 360))
        display(preview)


## 4. 按连续有效区间构造一个训练样本

只读取上述记录所在的区间，观察长度为 10，动作长度为 10，步长为 5。锚点为 t 时，观察取 t−9 至 t，动作取 t 至 t+9。窗口偏移左闭右开；长度不足直接跳过。

Prepared 的区间必须已保证原始帧连续。这里按 `valid_interval_id` 分组，避免跨越被排除的帧；不会重新划分 Train/Val/Test。实际训练应选择 train 集合，评估使用对应的 val/test 集合。


In [ ]:
interval_scan = table.scan().filter(
    (ds.field("valid_interval_id") == row["valid_interval_id"])
    & (ds.field("split") == row["split"])
)
windows = read_samples(
    interval_scan,
    episode_column="valid_interval_id",
    order_by="frame_index",
    window={
        "observation_state": (-9, 1),
        **{camera: (-9, 1) for camera in CAMERAS},
        "action": (0, 10),
    },
    stride=5,
    boundary="skip",
)
samples = windows.sort("anchor").limit(1).to_arrow().to_pylist()
if not samples:
    raise ValueError("当前区间不足以构造完整窗口，请选择另一条轨迹或调整窗口长度")
sample = samples[0]
t = sample["anchor"]

print("episode:", EPISODE_ID, "| split:", row["split"])
print("valid_interval_id:", sample["valid_interval_id"])
print(f"anchor: {t} | 观察帧: [{t - 9}, {t + 1}) | 动作帧: [{t}, {t + 10})")
print("state shape:", np.asarray(sample["observation_state"]).shape)
print("action shape:", np.asarray(sample["action"]).shape)
for camera in CAMERAS:
    print(f"{camera}: {len(sample[camera])} 张 JPEG 图片")


得到的是一个多模态训练样本：状态序列、四路 JPEG 序列和动作目标序列。尚未做归一化或堆叠 Tensor Batch，也没有训练模型。

## 导出为文档

运行前面四个代码单元格并保存 Notebook 后，在终端执行（需安装 Quarto）：

```bash
quarto render aloha_prepared_quickstart.ipynb --to docusaurus-md
```

转换使用保存的输出，不重新访问数据库。将生成的 Markdown 和引用的图片资源一起放入文档目录即可。仓库中的源文件未附带虚构的查询结果，需要在你的环境中执行后再发布。
